In [0]:
from pyspark.sql.functions import col, year, month, sum, count, countDistinct, round, avg, when, to_date

print("Iniciando a Camada Gold.")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

df_itens = spark.table("workspace.silver.fat_itens_pedidos")
df_pedidos = spark.table("workspace.silver.fat_pedido_total")
df_produtos = spark.table("workspace.silver.dim_produtos")
df_cotacao = spark.table("workspace.silver.dim_cotacao_dolar")
df_avaliacoes = spark.table("workspace.silver.fat_avaliacoes_pedidos")
df_vendedores = spark.table("workspace.silver.dim_vendedores")

df_base_vendas = df_itens.join(df_produtos, "id_produto", "left") \
    .join(df_pedidos.select("id_pedido", "data_pedido"), "id_pedido", "left") \
    .withColumn("ano_venda", year("data_pedido")) \
    .withColumn("mes_venda", month("data_pedido")) \
    .withColumn("data_cotacao", to_date("data_pedido")) \
    .join(df_cotacao, "data_cotacao", "left") \
    .withColumn("receita_item_usd", col("preco_BRL") / col("cotacao_compra"))

df_gold_comercial = df_base_vendas.groupBy("ano_venda", "mes_venda", "categoria_produto").agg(
    countDistinct("id_pedido").alias("total_pedidos"),
    count("id_item").alias("qtd_itens_vendidos"),
    round(sum("preco_BRL"), 2).alias("receita_total_brl"),
    round(sum("receita_item_usd"), 2).alias("receita_total_usd")
).withColumn("ticket_medio_brl", round(col("receita_total_brl") / col("total_pedidos"), 2))

df_gold_comercial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fat_vendas_comercial")
print("gold.fat_vendas_comercial gerada.")

df_base_avaliacoes = df_avaliacoes.join(df_itens, "id_pedido", "inner") \
    .join(df_produtos, "id_produto", "left") \
    .join(df_vendedores, "id_vendedor", "left")

df_gold_satisfacao = df_base_avaliacoes.groupBy("categoria_produto", "nome_vendedor", "estado").agg(
    count("id_avaliacao").alias("total_avaliacoes"),
    round(avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    sum(when(col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
    sum(when(col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
).withColumn("percentual_satisfacao", round((col("total_avaliacoes_positivas") / col("total_avaliacoes")) * 100, 2))

df_gold_satisfacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.gold.fat_avaliacoes_clientes")
print("gold.fat_avaliacoes_clientes gerada.")

print("\nRANKINGS COMERCIAIS\n")
df_top_produtos = df_base_vendas.groupBy("nome_produto", "categoria_produto").agg(count("id_item").alias("quantidade_vendida"))

print("1. Top 5 Produtos MAIS Vendidos:")
display(df_top_produtos.orderBy(col("quantidade_vendida").desc()).limit(5))

print("2. Top 5 Produtos MENOS Vendidos:")
display(df_top_produtos.orderBy(col("quantidade_vendida").asc()).limit(5))

print("\nRANKINGS DE QUALIDADE\n")
df_prod_avaliacao = df_base_avaliacoes.groupBy("nome_produto").agg(
    round(avg("nota_avaliacao"), 2).alias("nota"), count("id_avaliacao").alias("volume")
)
df_vend_avaliacao = df_base_avaliacoes.groupBy("nome_vendedor").agg(
    round(avg("nota_avaliacao"), 2).alias("nota"), count("id_avaliacao").alias("volume")
)

print("3. O Produto MAIS bem avaliado:")
display(df_prod_avaliacao.orderBy(col("nota").desc(), col("volume").desc()).limit(1))

print("4. O Produto MENOS bem avaliado:")
display(df_prod_avaliacao.orderBy(col("nota").asc(), col("volume").desc()).limit(1))

print("5. O Vendedor MAIS bem avaliado:")
display(df_vend_avaliacao.orderBy(col("nota").desc(), col("volume").desc()).limit(1))

print("6. O Vendedor MENOS bem avaliado:")
display(df_vend_avaliacao.orderBy(col("nota").asc(), col("volume").desc()).limit(1))

Iniciando a Camada Gold.
gold.fat_vendas_comercial gerada.
gold.fat_avaliacoes_clientes gerada.

RANKINGS COMERCIAIS

1. Top 5 Produtos MAIS Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804


2. Top 5 Produtos MENOS Vendidos:


nome_produto,categoria_produto,quantidade_vendida
Produto Genérico Preto,moveis_quarto,1
Item Básico Premium,fashion_calcados,1
Peça de Reposição Preto,telefonia_fixa,1
Item Básico Verde,industria_comercio_e_negocios,1
Peça de Reposição Plus,fashion_calcados,1



RANKINGS DE QUALIDADE

3. O Produto MAIS bem avaliado:


...

nome_produto,nota,volume
Conjunto de Pincéis Ultra,1.0,7


5. O Vendedor MAIS bem avaliado:


nome_vendedor,nota,volume
Luiz Otávio Abreu,5.0,32


6. O Vendedor MENOS bem avaliado:


nome_vendedor,nota,volume
Sra. Fernanda Santos,1.0,8
